# Combined Dataset
Performing analysis on the combined dataset consists of 98 repos from 5 benchmark datasets

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Configuration
# Path the consolidated CSV files. This file will be read and then overwritten
DATA_CSV_PATH = "/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/obj1_rd3.csv"
tercile_values = {}

In [4]:
def calculate_project_size(df):
    '''
    Calculates project_size ('small', 'medium', 'high') based on LoC terciles, grouped by langauge.
    '''
    print("Calculating 'project_size'...")

    def categorize(group):
        # Calculate the tercile boundaries for the 'LoC' column of the group
        tercile_1 = group['LoC'].quantile(1/3)
        tercile_2 = group['LoC'].quantile(2/3)

        language_name = group.name
        tercile_values[language_name] = {
            'small_medium_boundary': tercile_1,
            'medium_high_boundary': tercile_2
        }

        def assign_category(loc):
            if loc <= tercile_1:
                return 'small'
            elif loc <= tercile_2:
                return 'medium'
            else:
                return 'high'
        
        group['project_size'] = group['LoC'].apply(assign_category)
        return group
    
    # Apply the categorization function to each language group
    df = df.groupby('language', group_keys=False).apply(categorize)
    return df

In [5]:
def calculate_project_age(df, cutoff_year=2019):
    '''
    Calculates project_age ('new', 'mid-era') based on the median_bug_year and a specified cutoff year.
    '''

    def assign_age(year):
        if pd.isna(year):
            return 'unknown'
        return 'new' if year >= cutoff_year else 'mid-era'
    
    df['project_age'] = df['median_bug_year'].apply(assign_age)
    return df

In [6]:
def clean_dependencies(df):
    '''
    Cleans the num_dependencies column by replacing 0 values with the median of that repository's langauge group.
    '''
    print("Cleaning 'num_dependencies' column...")

    # Use transform to get the median for each language and align it with the original index
    lang_median = df.groupby('language')['num_dependencies'].transform('median')

    # Replace 0s with the calculated language-specific median
    df['num_dependencies'] = df['num_dependencies'].replace(0, np.nan).fillna(lang_median)

    return df


In [7]:
# Data prcoessing

try:
    # Load the dataset
    df = pd.read_csv(DATA_CSV_PATH)
    print(f"Successfully loaded {len(df)} records from '{DATA_CSV_PATH}'.")
except FileNotFoundError:
    print(f"Error: input file not found at '{DATA_CSV_PATH}'")
    

# Apply all three processing steps
df = calculate_project_size(df)
df = calculate_project_age(df)
df = clean_dependencies(df)

# Save the update DataFrame back to the same file
df.to_csv(DATA_CSV_PATH, index=False)

# Save the update DataFrame back to the same file
df.to_csv(DATA_CSV_PATH, index=False)

print("...Process COmplete....")
print(f"Updated Data Saved back to '{DATA_CSV_PATH}'")

# display a sample of the updated data
print("Sample of the final output:")
print(df[['repo_name', 'language', 'LoC', 'project_size', 'median_bug_year', 'project_age', 'num_dependencies']].head().to_string())

print("\n\n--- Calculated Tercile Boundaries (LoC) per Language ---")
for language, values in tercile_values.items():
    print(f"\nLanguage: {language}")
    print(f"  - 'small' <= {values['small_medium_boundary']:,.0f}")
    print(f"  - 'medium' <= {values['medium_high_boundary']:,.0f}")
    print(f"  - 'high' > {values['medium_high_boundary']:,.0f}")

Successfully loaded 98 records from '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/obj1_rd3.csv'.
Calculating 'project_size'...
Cleaning 'num_dependencies' column...
...Process COmplete....
Updated Data Saved back to '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/obj1_rd3.csv'
Sample of the final output:
                                      repo_name language      LoC project_size  median_bug_year project_age  num_dependencies
0                             quarkusio/quarkus     Java  1395911         high           2021.0         new          138257.0
1                       hannah-sten/texify-idea   Kotlin   130304         high           2021.0         new             328.0
2  horizontalsystems/unstoppable-wallet-android   Kotlin    39442        small           2019.0         new             191.0
3                   intellij-rust/intellij-rust   Kotlin   334728         high           2020.0         new     

/tmp/ipykernel_221399/725833137.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('language', group_keys=False).apply(categorize)


In [10]:
import pandas as pd

# Load the Parquet file
df = pd.read_parquet('/home/cs21d002_eashaan/PhD/Objective1/data/processed/bug_reports_clean.parquet')

# Display the column names
print("📦 Columns in the dataset:")
print(df.columns.tolist())
# Display the top 5 entries in the 'ground_truth_files' column
print("🔍 Preview of 'ground_truth_files' column:")
for i, row in df['ground_truth_files'].head(5).items():
    print(f"\nRow {i}:")
    print(row)


📦 Columns in the dataset:
['repo_name', 'bug_id', 'bug_report_text', 'ground_truth_files', 'creation_date', 'fix_date', 'pre_fix_commit_sha', 'fix_commit_sha', 'language', 'source_dataset', 'bug_report_url', 'fix_url']
🔍 Preview of 'ground_truth_files' column:

Row 0:
['changelogs/fragments/82359_assemble_diff.yml'
 'lib/ansible/plugins/action/__init__.py'
 'test/integration/targets/assemble/tasks/main.yml']

Row 1:
['changelogs/fragments/82353-ansible-sanity-examples.yml'
 'test/integration/targets/ansible-test-sanity-yamllint/aliases'
 'test/integration/targets/ansible-test-sanity-yamllint/ansible_collections/ns/col/plugins/inventory/inventory1.py'
 'test/integration/targets/ansible-test-sanity-yamllint/ansible_collections/ns/col/plugins/modules/module1.py'
 'test/integration/targets/ansible-test-sanity-yamllint/expected.txt'
 'test/integration/targets/ansible-test-sanity-yamllint/runme.sh'
 'test/lib/ansible_test/_util/controller/sanity/yamllint/yamllinter.py']

Row 2:
['changelogs/

In [11]:
# Helper function to extract extensions from a list of filenames
def extract_extensions(file_entry):
    # Convert to list if it's a string or numpy array-like
    if isinstance(file_entry, str):
        # Handle stringified list: remove brackets and split
        file_list = file_entry.strip("[]").replace("'", "").split()
    elif hasattr(file_entry, '__iter__'):
        file_list = list(file_entry)
    else:
        return set()
    
    # Extract extensions
    return set(f.split('.')[-1] for f in file_list if '.' in f)


# Extract extensions per row
df['file_exts'] = df['ground_truth_files'].apply(extract_extensions)

# Flag rows with multiple extensions
df['multi_ext_flag'] = df['file_exts'].apply(lambda x: len(x) > 1)

# Group by repo_name
grouped = df.groupby('repo_name')

# Analyze each repo
for repo, group in grouped:
    multi_ext_bugs = group[group['multi_ext_flag']]['bug_id'].tolist()
    if multi_ext_bugs:  # Only show repos with multiple extensions
        unique_exts = set().union(*group['file_exts'].tolist())
        
        print(f"\n📁 Repo: {repo}")
        print(f"🐞 Bug reports with multiple extensions: {len(multi_ext_bugs)}")
        print(f"🆔 Bug IDs: {multi_ext_bugs}")
        print(f"🧷 Unique extensions: {sorted(unique_exts)}")



📁 Repo: ansible/ansible
🐞 Bug reports with multiple extensions: 506
🆔 Bug IDs: ['82359', '82353', '82264', '82244', '82241', '82226', '82179', '82142', '82024', '82020', '82018', '81901', '81710', '81666', '81656', '81574', '81553', '81533', '81532', '81474', '81404', '81188', '81163', '81053', '80880', '80863', '80853', '80835', '80709', '80605', '80590', '80561', '80523', '80506', '80478', '80427', '80422', '80420', '80418', '80417', '80415', '80413', '80411', '80410', '80408', '80303', '80256', '80128', '80110', '80089', '79968', '79956', '79942', '79862', '79836', '79833', '79763', '79749', '79683', '79680', '79676', '79577', '79463', '79411', '79368', '79101', '79083', '79023', '78932', '78882', '78795', '78793', '78762', '78693', '78675', '78612', '78611', '78509', '78492', '78490', '78442', '78438', '78348', '78295', '78288', '78283', '78156', '78141', '78131', '78112', '78042', '77928', '77927', '77911', '77868', '77849', '77690', '77675', '77669', '77582', '77580', '77560', '